# MinIO + Azure Container Apps + Fabric walkthrough

Use this notebook to provision a test MinIO deployment on Azure Container Apps with Azure Verified Modules and capture the values needed for Microsoft Fabric shortcuts.

## 1) Validate Azure/Bicep tooling

Run the next cell to confirm Azure CLI and Bicep are ready inside Codespaces.

In [ ]:
!az version
!az bicep version

## 2) Review deployment settings

Set or override the deployment values before creating Azure resources. Environment variables let you customize names, location, image, or credentials without editing the notebook.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import secrets
import string
import subprocess
import tempfile

REPO_ROOT = Path('../')
TEMPLATE_FILE = REPO_ROOT / 'infra' / 'main.bicep'
TEMP_DIR = Path(tempfile.gettempdir()) / 'fabric-minio'
TEMP_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class CommandResult:
    success: bool
    stdout: str
    stderr: str
    returncode: int
    json_data: dict | None

class NotebookUtils:
    @staticmethod
    def run(command: str, success_message: str, failure_message: str) -> CommandResult:
        completed = subprocess.run(command, shell=True, capture_output=True, text=True)
        success = completed.returncode == 0
        if success:
            print(success_message)
        else:
            print(failure_message)

        stdout = completed.stdout.strip()
        stderr = completed.stderr.strip()
        json_data = None
        if stdout:
            try:
                json_data = json.loads(stdout)
            except json.JSONDecodeError:
                json_data = None

        if not success:
            raise RuntimeError(stderr or stdout or failure_message)

        return CommandResult(
            success=success,
            stdout=stdout,
            stderr=stderr,
            returncode=completed.returncode,
            json_data=json_data
        )

utils = NotebookUtils()

def random_suffix(length: int = 5) -> str:
    alphabet = string.ascii_lowercase + string.digits
    return ''.join(secrets.choice(alphabet) for _ in range(length))

def generate_secret(length: int = 24) -> str:
    alphabet = string.ascii_letters + string.digits
    return ''.join(secrets.choice(alphabet) for _ in range(length))

suffix = random_suffix()
timestamp = datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')
deployment_config = {
    'location': os.getenv('AZURE_LOCATION', 'italynorth'),
    'resource_group_name': os.getenv('RESOURCE_GROUP_NAME', f'rg-fabric-minio-{suffix}'),
    'deployment_name': os.getenv('DEPLOYMENT_NAME', f'fabric-minio-{timestamp}'),
    'log_analytics_workspace_name': os.getenv('LOG_ANALYTICS_WORKSPACE_NAME', f'log-fabric-minio-{suffix}'),
    'container_apps_environment_name': os.getenv('CONTAINER_APPS_ENVIRONMENT_NAME', f'cae-fabric-minio-{suffix}'),
    'container_app_name': os.getenv('CONTAINER_APP_NAME', f'minio-fabric-{suffix}'),
    'minio_image': os.getenv('MINIO_IMAGE', 'quay.io/minio/minio:latest'),
    'minio_root_user': os.getenv('MINIO_ROOT_USER', 'fabricminio'),
    'minio_root_password': os.getenv('MINIO_ROOT_PASSWORD', generate_secret()),
    'tags': {
        'workload': 'fabric-minio',
        'purpose': 'testing'
    }
}
deployment_config

## 3) Verify Azure login

Sign in with `az login` in a terminal first if this cell fails.

In [ ]:
!az account show --output table

## 4) Create the resource group

The notebook keeps resource group creation separate from the AVM deployment so you can reuse or delete the group independently.

In [ ]:
create_rg = [
    'az', 'group', 'create',
    '--name', deployment_config['resource_group_name'],
    '--location', deployment_config['location'],
    '--output', 'json'
]
print(' '.join(create_rg))
resource_group = json.loads(subprocess.run(create_rg, check=True, capture_output=True, text=True).stdout)
resource_group['id']

## 5) Validate the Bicep template

This catches local template issues before Azure deployment starts.

In [ ]:
!az bicep build --file $TEMPLATE_FILE

## 6) Deploy the Azure Verified Modules template

The next cell writes secure deployment parameters to `/tmp` and runs the group deployment. Outputs are retrieved in the following step.

In [ ]:
parameters_file = TEMP_DIR / 'main.parameters.json'
parameters_payload = {
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'location': {'value': deployment_config['location']},
        'logAnalyticsWorkspaceName': {'value': deployment_config['log_analytics_workspace_name']},
        'containerAppsEnvironmentName': {'value': deployment_config['container_apps_environment_name']},
        'containerAppName': {'value': deployment_config['container_app_name']},
        'minioImage': {'value': deployment_config['minio_image']},
        'minioRootUser': {'value': deployment_config['minio_root_user']},
        'minioRootPassword': {'value': deployment_config['minio_root_password']},
        'tags': {'value': deployment_config['tags']}
    }
}
parameters_file.write_text(json.dumps(parameters_payload, indent=2), encoding='utf-8')

# Run the deployment
output = utils.run(
    f"az deployment group create --name {deployment_config['deployment_name']} --resource-group {deployment_config['resource_group_name']} --template-file {TEMPLATE_FILE} --parameters {parameters_file}",
    f"Deployment '{deployment_config['deployment_name']}' succeeded",
    f"Deployment '{deployment_config['deployment_name']}' failed"
 )
output.success

## 7) Retrieve deployment outputs and copy the values into Microsoft Fabric

Use the deployment details query to fetch template outputs after deployment, then use the printed endpoint as the S3-compatible service URL for Fabric shortcuts and the generated access key pair as the connection credentials.

In [ ]:
# Obtain all deployment details and retrieve outputs
output = utils.run(
    f"az deployment group show --name {deployment_config['deployment_name']} --resource-group {deployment_config['resource_group_name']} --output json",
    f"Retrieved deployment: {deployment_config['deployment_name']}",
    f"Failed to retrieve deployment: {deployment_config['deployment_name']}"
 )

deployment_outputs = output.json_data.get('properties', {}).get('outputs', {}) if output.json_data else {}

if not deployment_outputs:
    raise RuntimeError('No deployment outputs were returned from Azure.')

connection_info = {
    'fabricShortcutServiceUrl': deployment_outputs['fabricShortcutServiceUrl']['value'],
    'fabricShortcutHost': deployment_outputs['fabricShortcutHost']['value'],
    'accessKeyId': deployment_config['minio_root_user'],
    'secretAccessKey': deployment_config['minio_root_password'],
    'containerAppResourceId': deployment_outputs['containerAppResourceId']['value']
}
connection_info

## 8) Upload sample files to MinIO

Run the next cell to install the S3 client (if needed), create a sample bucket, upload JSON and CSV sample files, and verify the uploaded object keys.

If your deployment endpoint uses a self-signed certificate, set `verify=False` in the client configuration for test-only scenarios.

In [ ]:
# Install dependency in the current kernel if needed
%pip install -q boto3

import json
from io import BytesIO

import boto3
from botocore.client import Config

endpoint_url = connection_info['fabricShortcutServiceUrl']
access_key = connection_info['accessKeyId']
secret_key = connection_info['secretAccessKey']

bucket_name = 'sample-data'

s3 = boto3.client(
    's3',
    endpoint_url=endpoint_url,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Create bucket if it does not already exist
existing_buckets = {bucket['Name'] for bucket in s3.list_buckets().get('Buckets', [])}
if bucket_name not in existing_buckets:
    s3.create_bucket(Bucket=bucket_name)

# Upload sample JSON file
sample_json = {
    'source': 'notebook',
    'rows': 3,
    'items': [{'id': 1}, {'id': 2}, {'id': 3}]
}
s3.put_object(
    Bucket=bucket_name,
    Key='samples/sample.json',
    Body=json.dumps(sample_json, indent=2).encode('utf-8'),
    ContentType='application/json'
)

# Upload sample CSV file
csv_buffer = BytesIO()
csv_buffer.write('id,name\n1,Ada\n2,Grace\n3,Linus\n'.encode('utf-8'))
csv_buffer.seek(0)

s3.put_object(
    Bucket=bucket_name,
    Key='samples/users.csv',
    Body=csv_buffer.read(),
    ContentType='text/csv'
)

uploaded = s3.list_objects_v2(Bucket=bucket_name, Prefix='samples/')
[item['Key'] for item in uploaded.get('Contents', [])]